In [1]:
army_urls = {
    "Main_Battle_Tanks": "https://www.globalfirepower.com/armor-tanks-total.php",
    "Armored_Fighting_Vehicles": "https://www.globalfirepower.com/armor-apc-total.php",
    "Self_Propelled_Artillery": "https://www.globalfirepower.com/armor-self-propelled-guns-total.php",
    "Towed_Artillery": "https://www.globalfirepower.com/armor-towed-artillery-total.php",
    "Multiple_Launch_Rocket_Systems": "https://www.globalfirepower.com/armor-mlrs-total.php"
}

In [2]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

In [4]:
def scrape_metric(url, metric_name):

    print(f"Scraping {metric_name}...")

    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print(f"Failed to load: {url}")
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    records = soup.find_all("div", class_="recordsetContainer")

    data = []

    for record in records:

        long_name = record.find("div", class_="longFormName")
        short_name = record.find("div", class_="shortFormName")
        value = record.find("div", class_="valueContainer")

        if value:

            if long_name:
                country = long_name.get_text(strip=True)
            elif short_name:
                country = short_name.get_text(strip=True)
            else:
                continue

            value = value.get_text(strip=True)

            data.append([country, value])

    df = pd.DataFrame(data, columns=["Country", metric_name])

    return df

In [5]:
dfs = []

for metric, url in army_urls.items():

    df = scrape_metric(url, metric)

    if df is not None:
        dfs.append(df)

    time.sleep(2)

Scraping Main_Battle_Tanks...
Scraping Armored_Fighting_Vehicles...
Scraping Self_Propelled_Artillery...
Scraping Towed_Artillery...
Scraping Multiple_Launch_Rocket_Systems...


In [6]:
army_df = dfs[0]

for df in dfs[1:]:

    army_df = army_df.merge(
        df,
        on="Country",
        how="outer"
    )

In [7]:
for col in army_df.columns[1:]:

    army_df[col] = (
        army_df[col]
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )

    army_df[col] = pd.to_numeric(
        army_df[col],
        errors="coerce"
    )

In [8]:
print(army_df.head())

       Country  Main_Battle_Tanks  Armored_Fighting_Vehicles  \
0  Afghanistan                  0                       3902   
1      Albania                  0                       1492   
2      Algeria               1485                      24920   
3       Angola                319                       2258   
4    Argentina                362                      22432   

   Self_Propelled_Artillery  Towed_Artillery  Multiple_Launch_Rocket_Systems  
0                         0                0                               0  
1                        10                0                             270  
2                       224              483                             188  
3                        25              575                             116  
4                        20              172                              27  


In [9]:
army_df.to_csv("army.csv", index=False)

print("Army dataset saved successfully!")

Army dataset saved successfully!


In [10]:
print("Shape:", army_df.shape)
print()

print(army_df.info())
print()

print(army_df.head(10))

Shape: (145, 6)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   Country                         145 non-null    object
 1   Main_Battle_Tanks               145 non-null    int64 
 2   Armored_Fighting_Vehicles       145 non-null    int64 
 3   Self_Propelled_Artillery        145 non-null    int64 
 4   Towed_Artillery                 145 non-null    int64 
 5   Multiple_Launch_Rocket_Systems  145 non-null    int64 
dtypes: int64(5), object(1)
memory usage: 6.9+ KB
None

       Country  Main_Battle_Tanks  Armored_Fighting_Vehicles  \
0  Afghanistan                  0                       3902   
1      Albania                  0                       1492   
2      Algeria               1485                      24920   
3       Angola                319                       2258   
4    Argentina         